In [ ]:
# @title
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

from sklearn.preprocessing import OneHotEncoder , OrdinalEncoder , StandardScaler
from sklearn.model_selection import train_test_split , GridSearchCV , KFold
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error , mean_absolute_error , r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


import joblib
import sklearn

url = "/content/drive/MyDrive/DataSets/dataset-6ab002b01f9ef078223946.csv"
df = pd.read_csv(url)

print(sklearn.__version__)

# 1. Data Understanding
  ## 1.1 Initial Data Inspection

In [ ]:
# @title
df.shape

In [ ]:
# @title
df.info()

In [ ]:
# @title
df.columns

In [ ]:
# @title
df.dtypes

In [ ]:
# @title
df.head()

In [ ]:
# @title
df.isna().sum()
# df.isna().mean() / 100
# 78 missing value in (Teacher_Quality)
# 90 missing value in (Parental_Education_Level)
# 67 missing value in (Distance_from_Home)

In [ ]:
# @title
df.duplicated().sum()

In [ ]:
# @title
df.describe()
# i think there is outlier in exam_Score // 101 ??!!

In [ ]:
# @title
df.describe(include=["object"])

# Distributions (histogrammes sur ExamScore et HoursStudied)

In [ ]:
# @title
fig , axes = plt.subplots(1 , 2 , figsize=(10 , 4))
axes[0].hist(df["Exam_Score"] , bins=10 , color="blue" , edgecolor = "black")
axes[0].set_title("Distributions ExamScore")
axes[0].set_xlabel("Exam_Score")
axes[0].set_ylabel("Frequency")

axes[1].hist(df["Hours_Studied"] , bins=10 , color="red" , edgecolor = "black")
axes[1].set_title("Distributions HoursStudied")
axes[1].set_xlabel("Hours_Studied")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.show()

In [ ]:
# @title
corr_matrix = df.corr(numeric_only=True)
# corr_matrix

plt.figure(figsize=(10 , 8))

sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1 , vmax=1,
    linewidths=0.5
)

plt.title("Matrix Student Exam Prediction", fontsize = 14)
plt.tight_layout()
plt.show()

In [ ]:
# @title
plt.figure(figsize=(6 , 4))
sns.boxplot(x = df["Exam_Score"] , color="blue")

plt.title("Boxplot of Exam Score")
plt.xlabel("Exam Score")

plt.show()

In [ ]:
# @title
Q1 = df["Exam_Score"].quantile(0.25)
Q3 = df["Exam_Score"].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print(Q1)
print(Q3)
print(IQR)

print(lower)
print(upper)

In [ ]:
# @title
(df["Exam_Score"] > upper).sum()
(df["Exam_Score"] < lower).sum()


In [ ]:
# @title
df[np.abs(stats.zscore(df["Exam_Score"])) > 3]["Exam_Score"].count()

In [ ]:
# @title
df.loc[(df["Exam_Score"] > upper) | (df["Exam_Score"] < lower) , "Exam_Score"]

#2. Data Cleaning
  # 2.1 Handling Missing Values

In [ ]:
# @title
df["Teacher_Quality"] = df["Teacher_Quality"].fillna(df["Teacher_Quality"].mode()[0])
df["Parental_Education_Level"] = df["Parental_Education_Level"].fillna(df["Parental_Education_Level"].mode()[0])
df["Distance_from_Home"] = df["Distance_from_Home"].fillna(df["Distance_from_Home"].mode()[0])

# df["Teacher_Quality"].isna().sum()
# df["Parental_Education_Level"].isna().sum()
# df["Distance_from_Home"].isna().sum()

# 2.3 Removing Outliers

In [ ]:
# @title
df = df[df['Exam_Score'].between(lower , upper)]
# (df["Exam_Score"] > upper).sum()
# (df["Exam_Score"] < lower).sum()

# 3 Feature Engineering and Encoding
# 3.1 Nominal Variables → One-Hot Encoding

In [ ]:
print(df["Gender"].value_counts(dropna= False))
print(df["School_Type"].value_counts(dropna= False))
print(df["Extracurricular_Activities"].value_counts(dropna= False))
print(df["Internet_Access"].value_counts(dropna= False))
print(df["Learning_Disabilities"].value_counts(dropna= False))
print(df["Peer_Influence"].value_counts(dropna= False))

In [ ]:
# encoder = OneHotEncoder(drop="first" , sparse_output=False)
# encoder.fit_transform(df[["Gender"]])
# encoder.get_feature_names_out(["Gender"])

nominal= [
    "Gender",
    "School_Type",
    "Extracurricular_Activities",
    "Internet_Access",
    "Learning_Disabilities",
    "Peer_Influence"
]

oneHot_encoder = OneHotEncoder(drop="first" , sparse_output=False)

encoder = oneHot_encoder.fit_transform(df[nominal])

# encoder
nominal_name = oneHot_encoder.get_feature_names_out(nominal)

# nominal_name

# oneHot_encoder.categories_

In [ ]:
nominal_df = pd.DataFrame(
    encoder,
    columns=nominal_name,
    index= df.index
)

nominal_df.head()

# 3.2 Ordinal Variables → Ordinal Encoder

In [ ]:
print(df["Parental_Involvement"].value_counts(dropna= False))
print(df["Access_to_Resources"].value_counts(dropna= False))
print(df["Motivation_Level"].value_counts(dropna= False))
print(df["Family_Income"].value_counts(dropna= False))
print(df["Teacher_Quality"].value_counts(dropna= False))
print(df["Parental_Education_Level"].value_counts(dropna= False))
print(df["Distance_from_Home"].value_counts(dropna= False))

In [ ]:
ordinal= [
    "Parental_Involvement",
    "Access_to_Resources",
    "Motivation_Level",
    "Family_Income",
    "Teacher_Quality",
    "Parental_Education_Level",
    "Distance_from_Home"
]

# ---------------------------------

o_categories = [
    ["Low", "Medium", "High"],
    ["Low", "Medium", "High"],
    ["Low", "Medium", "High"],
    ["Low", "Medium", "High"],
    ["Low", "Medium", "High"],
    ["High School", "College", "Postgraduate"],
    ["Near", "Moderate", "Far"]
]

ordinal_encoder = OrdinalEncoder(
    categories = o_categories
)

o_encoder = ordinal_encoder.fit_transform(df[ordinal])

# ordinal_encoder

ordinal_names = ordinal_encoder.get_feature_names_out(ordinal)

ordinal_names


In [ ]:
ordinal_df = pd.DataFrame(
    o_encoder,
    columns=ordinal_names,
    index=df.index
)

ordinal_df.head()

# 3.3 Final Feature Dataset.

In [ ]:
numerical = [
    "Hours_Studied",
    "Attendance",
    "Sleep_Hours",
    "Previous_Scores",
    "Tutoring_Sessions",
    "Physical_Activity"
]

X = pd.concat(
    [
        df[numerical],
        nominal_df,
        ordinal_df
    ],

    axis=1
)

# X.head()

y = df["Exam_Score"]

# y.shape , x.shape

# Découpage 80% / 20% avec traintestsplit

In [ ]:
X_train , X_test , y_train , y_test = train_test_split(
    X , y ,
    test_size=0.20,
    random_state= 42
    )

# X_train.shape, X_test.shape, y_train.shape, y_test.shape

num_colums = [
    "Hours_Studied",
    "Attendance",
    "Sleep_Hours",
    "Previous_Scores",
    "Tutoring_Sessions",
    "Physical_Activity"
]

scaler = StandardScaler()

x_train_scaler = scaler.fit_transform(X_train[num_colums])

x_test_scaler = scaler.transform(X_test[num_colums])

# x_train_scaler.shape, x_test_scaler.shape

# x_train_scaler[:5]

X_train_num = pd.DataFrame(
    x_train_scaler,
    columns = num_colums,
    index = X_train.index
)

X_test_num = pd.DataFrame(
    x_test_scaler,
    columns = num_colums,
    index = X_test.index
)

# X_train_num.head()



In [ ]:
x_train_ord = ordinal_df.loc[X_train.index]
x_test_ord = ordinal_df.loc[X_test.index]

x_train_nom = nominal_df.loc[X_train.index]
x_test_nom = nominal_df.loc[X_test.index]

X_train_final = pd.concat(
    [X_train_num , x_train_ord, x_train_nom],
    axis=1
)

X_test_final = pd.concat(
    [X_test_num , x_test_ord , x_test_nom],
    axis=1
)

X_train_final.shape, X_test_final.shape

#5. Model Training
#5.1 Linear Regression.

In [ ]:
linear_model = LinearRegression()

linear_model.fit(
    X_train_final,
    y_train
)

y_pred = linear_model.predict(X_test_final)

y_pred.shape

#5.2 Model Evaluation

In [ ]:
rmse_linear = root_mean_squared_error(
    y_test,
    y_pred,
)

rmse_linear

In [ ]:
mae_linear = mean_absolute_error(
    y_test,
    y_pred
)

mae_linear

In [ ]:
r2_linear = r2_score(
    y_test,
    y_pred
)

r2_linear

#5.2 Random Forest

In [ ]:
rand = RandomForestRegressor(random_state=42)

rand.fit(
    X_train_final,
    y_train
)

rand_pred = rand.predict(
    X_test_final
)

rand_pred.shape

# 5.2 Model Evaluation

In [ ]:
rand_rmse = root_mean_squared_error(
    y_test,
    rand_pred
)

rand_rmse

In [ ]:
rand_mae = mean_absolute_error(
    y_test,
    rand_pred
)

rand_mae

In [ ]:
rand_r2 = r2_score(
    y_test,
    rand_pred
)

rand_r2

5.2 SVR

In [ ]:
svr = SVR()

svr.fit(
    X_train_final,
    y_train
)

svr_pred = svr.predict(X_test_final)

In [ ]:
svr_rmse = root_mean_squared_error(
    y_test,
    svr_pred
)

svr_rmse

In [ ]:
svr_mae = mean_absolute_error(
    y_test,
    svr_pred
)

svr_mae

In [ ]:
svr_r2 = r2_score(
    y_test,
    svr_pred
)

svr_r2

#Hyperparameter Optimization

In [ ]:
result = pd.DataFrame({
    "Model" : ["Linear Regression" , "Random Forest" , "SVR"],
    "Rmse"  : [rmse_linear , rand_rmse , svr_rmse],
    "Mae"   : [mae_linear , rand_mae , svr_mae],
    "R2"    : [r2_linear , rand_r2 , svr_r2]
})

result

#Linear Regression Hyperparameter Tuning

In [ ]:
linear_params = {
    "fit_intercept" : [True , False],
    "positive" : [True , False]
}

linear_grid = GridSearchCV(
    LinearRegression(),
    param_grid = linear_params,
    cv = 3,
    scoring = "r2"
)

linear_grid.fit(
    X_train_final,
    y_train
)

linear_grid.best_params_

# Random Forest Hyperparameter Tuning

In [ ]:
# rf_params = {
#     "n_estimators": [100 , 200],
#     "max_depth" : [None , 10 , 20],
#     "min_simples_split" : [2 , 5]
# }

# kf = KFold(
#     n_splits = 3,
#     shuffle = True,
#     random_state=42
# )

# rf_grid = GridSearchCV(
#     RandomForestRegressor(random_state = 42),
#     param_grid = rf_params,
#     cv = kf,
#     scoring = "r2"
# )

# rf_grid.fit(
#     X_train_final,
#     y_train
# )

# rf_grid.best_params_

# SVR Hyperparameter Tuning

In [ ]:
# svr_params = {
#     "C" : [0.1 , 1 , 10 , 100],
#     "kernel" : ["linear" , "rbf"],
#     "epsilon" : [0.01 , 0.1 , 0.2]
# }

# svr_grid = GridSearchCV(
#     # SVR(),
#     # param_grid = svr_params,
#     # cv = kf,
#     # scoring = "r2"
# )

# svr_grid.fit(
#     # X_train_final,
#     # y_train
# )

# svr_grid.best_params_

#Optimized Models Evaluation on Test Data


In [ ]:
best_linear = linear_grid.best_estimator_
# best_rf = rf_grid.best_estimator_
# best_svr = svr_grid.best_estimator_

linear_opt_pred = best_linear.predict(X_test_final)
# rf_opt_pred = best_rf.predict(X_test_final)
# svr_opt_pred = best_svr.predict(X_test_final)

linear_opt_rmse = root_mean_squared_error(y_test , linear_opt_pred)
linear_opt_mae = mean_absolute_error(y_test , linear_opt_pred)
linear_opt_r2 =  r2_score(y_test, linear_opt_pred)

# rf_opt_rmse = root_mean_squared_error(y_test, rf_opt_pred)
# rf_opt_mae = mean_absolute_error(y_test, rf_opt_pred)
# rf_opt_r2 = r2_score(y_test, rf_opt_pred)

# svr_opt_rmse = root_mean_squared_error(y_test, svr_opt_pred)
# svr_opt_mae = mean_absolute_error(y_test, svr_opt_pred)
# svr_opt_r2 = r2_score(y_test, svr_opt_pred)



In [ ]:
# optimized_results = pd.DataFrame({
#     "Model": ["Linear Regression", "Random Forest", "SVR"],
#     "RMSE": [linear_opt_rmse, rf_opt_rmse, svr_opt_rmse],
#     "MAE": [linear_opt_mae, rf_opt_mae, svr_opt_mae],
#     "R2": [linear_opt_r2, rf_opt_r2, svr_opt_r2]
# })

# optimized_results

#Residual Analysis for Linear Regression

In [ ]:
residuals = y_test - linear_opt_pred

residuals.describe()

#Saving the Trained Model (Model Serialization)

In [ ]:
joblib.dump(best_linear , "student_exam_score_model.pkl")

In [ ]:
!ls -lh student_exam_score_model.pkl

In [ ]:
preprocessor = ColumnTransformer(
    transformers = [
        ("num" , StandardScaler() , numerical),
        ("ord" , OrdinalEncoder(categories = o_categories) , ordinal ),
        ("nom" , OneHotEncoder(drop="first" , sparse_output=False, handle_unknown="ignore") , nominal)

    ]
)

final_pipeline = Pipeline(
    steps = [
        ("preprocessor" , preprocessor),
        ("model" , LinearRegression(
            fit_intercept=True,
            positive=False
            ))
    ]
)

In [ ]:
X_raw = df.drop(columns=["Exam_Score"])

X_raw_train = X_raw.loc[X_train.index]
X_raw_test = X_raw.loc[X_test.index]

final_pipeline.fit(X_raw_train, y_train)

In [ ]:
final_pred = final_pipeline.predict(X_raw_test)

final_rmse = root_mean_squared_error(y_test, final_pred)
final_mae = mean_absolute_error(y_test, final_pred)
final_r2 = r2_score(y_test, final_pred)

print("RMSE:", final_rmse)
print("MAE:", final_mae)
print("R²:", final_r2)

In [ ]:
joblib.dump(
    final_pipeline,
    "student_exam_score_pipeline.pkl"
)

In [ ]:
!ls -lh student_exam_score_pipeline.pkl

In [ ]:
from google.colab import files

files.download("student_exam_score_pipeline.pkl")